# 10 – Hedging Backtest

Out-of-sample hedging experiment on the 2025 test year.

**Claim**: short indicator quanto $H = \mathbf{1}_{S_T > K_S}\,\mathbf{1}_{Y_T > K_Y}$
with 1-month maturity, at-the-money strikes, and weekly rebalancing.

**Hedge instrument**: one-month energy forward.

**Strategies compared**:
1. *Unhedged*: no forward hedge
2. *OU hedge*: delta from calibrated CARMA(1,0) (OU) model
3. *CARMA hedge*: GKW hedge ratio $\xi_t$ via finite differences on Fourier price

**Metric**: variance reduction ratio (VRR) and RMSE of weekly tracking error.

Inputs:  `results/carma_temp_mle.json`, `results/carma_price_mle.json`,
         `results/levy_params_*.json`, `results/coupling_params.json`,
         `data/deseasonalised/temp_resid.csv`, `data/deseasonalised/price_resid.csv`
         (test split: 2025)  
Outputs: `results/hedging_backtest.csv`, `figures/hedging_backtest.png`

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import (
    RES_DIR, FIG_DIR, DT_YEARS, HOURS_PER_YEAR,
    TEMP_RESID_TRAIN_CSV, PRICE_RESID_TRAIN_CSV,
)
from carma_utils import (
    build_companion, load_params, kalman_filter,
    compute_forward_price,
    fourier_price_indicator_quanto,
    compute_hedge_ratio_fd,
)
from pathlib import Path

# ── Load fitted parameters ────────────────────────────────────────────────────
params_temp  = load_params(RES_DIR / 'carma_temp_mle.json')
params_price = load_params(RES_DIR / 'carma_price_mle.json')

with open(RES_DIR / 'levy_params_temperature.json') as f:
    levy_temp  = json.load(f)
with open(RES_DIR / 'levy_params_logprice.json') as f:
    levy_price = json.load(f)
with open(RES_DIR / 'coupling_params.json') as f:
    coupling   = json.load(f)

nig_Y = levy_temp['nig']
nig_X = levy_price['nig']
gamma0 = float(coupling['gamma0'])

# ── Build state-space matrices ────────────────────────────────────────────────
def build_system(params):
    a = np.array(params['a_coeffs'])
    b = np.array(params['b_coeffs'])
    p = len(a)
    A = build_companion(a)
    B = np.zeros((p, 1)); B[-1, 0] = params['sigma']
    c = np.zeros(p); c[:len(b)] = b
    return A, B, c, p

A_X, B_X, c_X, p_X = build_system(params_price)
A_Y, B_Y, c_Y, p_Y = build_system(params_temp)
Gamma = gamma0 * B_X

# OU (CARMA(1,0)) benchmark: fit the OU model separately
# Use first eigenvalue as mean-reversion speed, same sigma for simplicity
lambda_ou = float(params_price['eigenvalues_real'][0])  # most negative
A_OU = np.array([[lambda_ou]])
B_OU = np.array([[params_price['sigma']]])
c_OU = np.array([1.0])
Gamma_OU = gamma0 * B_OU

print('System matrices loaded.')
print(f'  CARMA: p_X={p_X}, p_Y={p_Y}')
print(f'  OU eigenvalue: {lambda_ou:.2f} yr^-1  (τ = {-1/lambda_ou:.4f} yr = {-1/(lambda_ou*DT_YEARS):.1f} h)')

## 1.  Load and filter test data (2025)

In [ ]:
DESEAS_DIR = Path('../data/deseasonalised')

# Load full residual series and extract test window (2025)
def load_test_residuals(path, col):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    y  = df[col].dropna().to_numpy(dtype=float)
    # Test window: last 8760 observations (1 year)
    return y[-8760:]

y_temp_test  = load_test_residuals(DESEAS_DIR / 'temp_resid.csv',      'temp_deseasoned')
y_price_test = load_test_residuals(DESEAS_DIR / 'price_logresid.csv',  'logprice_deseasoned')

# Align to common length
n_test = min(len(y_temp_test), len(y_price_test))
y_temp_test  = y_temp_test[-n_test:]
y_price_test = y_price_test[-n_test:]

t_test = np.arange(n_test, dtype=float) * DT_YEARS
print(f'Test observations: {n_test:,} hours')

# Run Kalman filter on test data to get state estimates
res_price_test = kalman_filter(
    t_years=t_test, y=y_price_test,
    a_coeffs=params_price['a_coeffs'],
    b_coeffs=params_price['b_coeffs'],
    sigma=params_price['sigma'],
)
res_temp_test = kalman_filter(
    t_years=t_test, y=y_temp_test,
    a_coeffs=params_temp['a_coeffs'],
    b_coeffs=params_temp['b_coeffs'],
    sigma=params_temp['sigma'],
)

x_hat_X = res_price_test['x_filt']   # (n_test, p_X)
x_hat_Y = res_temp_test['x_filt']    # (n_test, p_Y)
print(f'Kalman filter complete. State shapes: X={x_hat_X.shape}, Y={x_hat_Y.shape}')

## 2.  Rolling weekly hedge backtest

At each weekly rebalancing date $t_k$:
1. Kalman state $Z_{t_k}^X$, $Z_{t_k}^Y$ from filter
2. Forward price $F(t_k, T_k)$ where $T_k = t_k + 1$\ month
3. Option value $V(t_k)$ via Fourier
4. Hedge ratio $\xi(t_k)$ via finite differences
5. Record weekly PnL $\Pi_{k+1} = (V_{k+1} - V_k) - \xi_k(F_{k+1} - F_k)$

In [ ]:
WEEKS_PER_YEAR = 52
HOURS_PER_WEEK = HOURS_PER_YEAR / WEEKS_PER_YEAR
TAU_CONTRACT   = 1.0 / 12.0   # 1-month maturity

# Weekly rebalancing dates
step = int(HOURS_PER_WEEK)
rebal_idx = np.arange(0, n_test - step, step)
n_rebal   = len(rebal_idx)
print(f'Rebalancing dates: {n_rebal} weeks')

# Storage
V_carma = np.zeros(n_rebal)
F_carma = np.zeros(n_rebal)
xi_carma = np.zeros(n_rebal)

V_ou    = np.zeros(n_rebal)
F_ou    = np.zeros(n_rebal)
xi_ou   = np.zeros(n_rebal)

# Strikes: set at start of backtest (ATM at t=0)
Z_X_init = x_hat_X[0]
Z_Y_init = x_hat_Y[0]
TAU_EVAL  = TAU_CONTRACT

K_S = compute_forward_price(
    TAU_EVAL, Z_X_init, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
    nig_X, nig_Y
)
K_Y = float(c_Y @ Z_Y_init)
print(f'Contract strikes: K_S = {K_S:.4f}, K_Y = {K_Y:.4f}')

for k, idx in enumerate(rebal_idx):
    if k % 10 == 0:
        print(f'  Week {k}/{n_rebal} …', end='\r')

    Z_X = x_hat_X[idx]
    Z_Y = x_hat_Y[idx]

    # CARMA model
    F_carma[k] = compute_forward_price(
        TAU_EVAL, Z_X, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
        nig_X, nig_Y
    )
    V_carma[k] = fourier_price_indicator_quanto(
        TAU_EVAL, Z_X, Z_Y, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
        nig_X, nig_Y, K_S=K_S, K_Y=K_Y,
    )
    xi_carma[k] = compute_hedge_ratio_fd(
        TAU_EVAL, Z_X, Z_Y, A_X, c_X, B_X, A_Y, c_Y, B_Y, Gamma,
        nig_X, nig_Y, K_S=K_S, K_Y=K_Y,
    )

    # OU model (CARMA(1,0) with p_X=1, p_Y=1)
    Z_X_ou = np.array([float(c_X @ Z_X)])  # project to 1D
    Z_Y_ou = np.array([float(c_Y @ Z_Y)])
    F_ou[k] = compute_forward_price(
        TAU_EVAL, Z_X_ou, A_OU, c_OU, B_OU,
        np.array([[params_temp['eigenvalues_real'][0]]]),
        c_OU, B_OU, Gamma_OU, nig_X, nig_Y
    )
    V_ou[k] = fourier_price_indicator_quanto(
        TAU_EVAL, Z_X_ou, Z_Y_ou,
        A_OU, c_OU, B_OU,
        np.array([[params_temp['eigenvalues_real'][0]]]),
        c_OU, B_OU, Gamma_OU,
        nig_X, nig_Y, K_S=K_S, K_Y=K_Y,
    )
    xi_ou[k] = compute_hedge_ratio_fd(
        TAU_EVAL, Z_X_ou, Z_Y_ou,
        A_OU, c_OU, B_OU,
        np.array([[params_temp['eigenvalues_real'][0]]]),
        c_OU, B_OU, Gamma_OU,
        nig_X, nig_Y, K_S=K_S, K_Y=K_Y,
    )

print(f'\nBacktest complete.')

## 3.  Compute hedge errors and performance metrics

In [ ]:
# Weekly changes
dV_carma = np.diff(V_carma)     # value change (short position)
dF_carma = np.diff(F_carma)     # forward change
dV_ou    = np.diff(V_ou)
dF_ou    = np.diff(F_ou)

# Hedge PnL: Pi = -dV + xi * dF (short option, long hedge)
pnl_unhedged = -dV_carma
pnl_carma    = -dV_carma + xi_carma[:-1] * dF_carma
pnl_ou       = -dV_ou    + xi_ou[:-1]    * dF_ou

var_unhedged = float(np.var(pnl_unhedged))
var_carma    = float(np.var(pnl_carma))
var_ou       = float(np.var(pnl_ou))

vrr_carma = 1.0 - var_carma / var_unhedged
vrr_ou    = 1.0 - var_ou    / var_unhedged

rmse_unhedged = float(np.sqrt(np.mean(pnl_unhedged**2)))
rmse_carma    = float(np.sqrt(np.mean(pnl_carma**2)))
rmse_ou       = float(np.sqrt(np.mean(pnl_ou**2)))

print('Hedging performance:')
print(f'  Unhedged      : Var={var_unhedged:.6f}  RMSE={rmse_unhedged:.6f}  VRR=---')
print(f'  OU hedge      : Var={var_ou:.6f}    RMSE={rmse_ou:.6f}    VRR={vrr_ou:.1%}')
print(f'  CARMA hedge   : Var={var_carma:.6f}  RMSE={rmse_carma:.6f}  VRR={vrr_carma:.1%}')

results_df = pd.DataFrame({
    'week': range(len(dV_carma)),
    'pnl_unhedged': pnl_unhedged,
    'pnl_ou':       pnl_ou,
    'pnl_carma':    pnl_carma,
})
results_df.to_csv(RES_DIR / 'hedging_backtest.csv', index=False)
print(f'Saved → {RES_DIR}/hedging_backtest.csv')

# Summary table
summary = pd.DataFrame([
    {'Strategy': 'Unhedged',        'VRR': None,       'RMSE': rmse_unhedged,
     'Mean_PnL': pnl_unhedged.mean()},
    {'Strategy': 'OU hedge',        'VRR': vrr_ou,     'RMSE': rmse_ou,
     'Mean_PnL': pnl_ou.mean()},
    {'Strategy': 'CARMA(3,1) hedge','VRR': vrr_carma,  'RMSE': rmse_carma,
     'Mean_PnL': pnl_carma.mean()},
])
summary.to_csv(RES_DIR / 'hedging_summary.csv', index=False)
print(summary.to_string(index=False, float_format=lambda x: f'{x:.5f}' if x else '---'))

## 4.  Cumulative PnL plot

In [ ]:
weeks = np.arange(len(pnl_unhedged))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative PnL
ax = axes[0]
ax.plot(weeks, np.cumsum(pnl_unhedged), 'b-',  lw=1.5, alpha=0.7, label='Unhedged')
ax.plot(weeks, np.cumsum(pnl_ou),       'g--', lw=1.5, label='OU hedge')
ax.plot(weeks, np.cumsum(pnl_carma),    'r-',  lw=2.0, label='CARMA hedge')
ax.fill_between(weeks,
                np.cumsum(pnl_unhedged) - np.std(pnl_unhedged),
                np.cumsum(pnl_unhedged) + np.std(pnl_unhedged),
                alpha=0.15, color='b')
ax.set_xlabel('Week')
ax.set_ylabel('Cumulative PnL')
ax.set_title('Cumulative hedge PnL — 2025 backtest')
ax.legend()
ax.axhline(0, color='k', lw=0.8, ls=':')

# Rolling variance (13-week window)
ax = axes[1]
window = 13
for pnl, label, ls in [
    (pnl_unhedged, 'Unhedged', '-'),
    (pnl_ou,       'OU hedge',  '--'),
    (pnl_carma,    'CARMA hedge', '-'),
]:
    roll_var = pd.Series(pnl).rolling(window).var()
    ax.plot(weeks, roll_var, ls=ls, label=label)
ax.set_xlabel('Week')
ax.set_ylabel('Rolling 13-week PnL variance')
ax.set_title('Rolling hedge error variance')
ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'hedging_backtest.png', dpi=150)
plt.show()
print('Saved hedging_backtest.png')

print(f'\nFinal summary:')
print(f'  CARMA VRR = {vrr_carma:.1%}  vs  OU VRR = {vrr_ou:.1%}')
print(f'  VRR improvement: {(vrr_carma - vrr_ou)*100:.1f} percentage points')